# PoC V2.0 — Fases 3 y 4: Modelo Predictivo + ADAPT-VQE Restringido

Este notebook implementa las fases finales de la arquitectura híbrida GNN-HVA:

- **Fase 3**: MLP entrenado con PyTorch para predecir $\theta_{opt}$ a partir de $h$, con validación física vía `StatevectorEstimator`.
- **Fase 4**: Despliegue sobre un Hamiltoniano no visto ($h=1.05$, cerca del punto crítico), usando el MLP como warm-start para un ADAPT-VQE restringido ($\leq 2$ iteraciones), con caracterización de fase mediante observables locales.

**Dependencias**: `numpy`, `torch`, `matplotlib`, `qiskit>=2.0`, `qiskit-algorithms>=0.3`, `scipy`

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt

from qiskit.quantum_info import SparsePauliOp, Statevector
from qiskit.circuit import QuantumCircuit, ParameterVector
from qiskit.primitives import StatevectorEstimator

# Reproducibilidad
torch.manual_seed(42)
np.random.seed(42)

## Funciones auxiliares (replicadas de Fase 1-2 para independencia del notebook)

In [ ]:
def build_tfim_hamiltonian(N, J, h):
    """H = -J Σ ZᵢZᵢ₊₁ - h Σ Xᵢ"""
    terms = [("ZZ", [i, i+1], -J) for i in range(N-1)] + [("X", [i], -h) for i in range(N)]
    return SparsePauliOp.from_sparse_list(terms, num_qubits=N)

def create_hva_circuit(N, p):
    """HVA: e^{-iθ_x H_X} · e^{-iθ_zz H_ZZ} por capa sobre |+⟩^N. Factor 2θ para RZZ/RX."""
    qc = QuantumCircuit(N)
    qc.h(range(N))  # |+⟩^N: ground state paramagnético (h → ∞)
    theta = ParameterVector('θ', 2 * p)
    for layer in range(p):
        for i in range(N - 1):
            qc.rzz(2 * theta[layer * 2], i, i + 1)
        for i in range(N):
            qc.rx(2 * theta[layer * 2 + 1], i)
    return qc, theta

def build_local_observables(N):
    """Observables locales: ⟨Xᵢ⟩ por sitio, ⟨ZᵢZᵢ₊₁⟩ por enlace."""
    ops_X = [SparsePauliOp.from_sparse_list([("X", [i], 1.0)], num_qubits=N) for i in range(N)]
    ops_ZZ = [SparsePauliOp.from_sparse_list([("ZZ", [i, i+1], 1.0)], num_qubits=N) for i in range(N-1)]
    return ops_X, ops_ZZ

---
# FASE 3: Modelo Predictivo (PyTorch MLP)

## Step 3.1: Ingesta de datos y preprocesamiento

In [ ]:
data = np.load("phase1_phase2_tfim_N6_p2.npz", allow_pickle=True)

N = int(data["n_qubits"])
J = float(data["J"])
p_layers = int(data["p_layers"])
h_values = data["h_values"]
ground_energies = data["ground_energies"]
theta_opt_all = data["theta_opt"]          # shape: (21, 4)
mag_x_exact = data["mag_x"]
corr_zz_exact = data["corr_zz"]

# Inputs X: h values → (21, 1), Targets Y: θ_opt → (21, 4)
X = torch.tensor(h_values, dtype=torch.float32).unsqueeze(1)
Y = torch.tensor(theta_opt_all, dtype=torch.float32)

n_params = Y.shape[1]
print(f"Dataset: {len(X)} puntos, input_dim=1, output_dim={n_params}")
print(f"h ∈ [{h_values[0]:.1f}, {h_values[-1]:.1f}], θ_opt shape: {theta_opt_all.shape}")

## Step 3.2: Arquitectura del MLP

In [ ]:
class HVAPredictor(nn.Module):
    """MLP: h → θ_pred ∈ ℝ^(2p)"""
    def __init__(self, n_out):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(1, 16), nn.ReLU(),
            nn.Linear(16, 16), nn.ReLU(),
            nn.Linear(16, n_out),
        )

    def forward(self, x):
        return self.net(x)

model = HVAPredictor(n_params)
print(f"Parámetros del modelo: {sum(p.numel() for p in model.parameters())}")
print(model)

## Step 3.3: Entrenamiento híbrido (MSE + validación energética)

Cada 100 épocas, se evalúa la energía cuántica $E(\theta_{pred})$ vía `StatevectorEstimator` para verificar que los ángulos predichos retienen significado físico.

In [ ]:
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=200, factor=0.5)
loss_fn = nn.MSELoss()
estimator = StatevectorEstimator()
hva_qc, theta_params = create_hva_circuit(N, p_layers)

n_epochs = 2000
loss_history = []
energy_check_epochs = []
energy_check_errors = []  # max |E_pred - E_exact| across all h

# Validación: 5 puntos de entrenamiento + 1 punto de interpolación (h=0.95, no visto)
check_indices = [0, 5, 10, 15, 20]  # h = 0.0, 0.5, 1.0, 1.5, 2.0
h_interp = 0.95  # punto de interpolación no visto
H_interp = build_tfim_hamiltonian(N, J, h_interp)
e_interp_exact = np.linalg.eigh(H_interp.to_matrix())[0][0]

print("Entrenamiento del MLP...")
for epoch in range(n_epochs):
    model.train()
    pred = model(X)
    loss = loss_fn(pred, Y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    loss_history.append(loss.item())
    scheduler.step(loss.item())

    # Validación física cada 100 épocas
    if (epoch + 1) % 100 == 0:
        model.eval()
        max_e_error = 0.0
        with torch.no_grad():
            for idx in check_indices:
                h = h_values[idx]
                theta_p = model(X[idx:idx+1]).numpy().flatten()
                H = build_tfim_hamiltonian(N, J, h)
                bound = hva_qc.assign_parameters(theta_p)
                e_p = estimator.run([(bound, H)]).result()[0].data.evs
                max_e_error = max(max_e_error, abs(e_p - ground_energies[idx]))

            # Interpolación: h=0.95 (no visto)
            theta_interp = model(torch.tensor([[h_interp]], dtype=torch.float32)).numpy().flatten()
            bound_interp = hva_qc.assign_parameters(theta_interp)
            e_interp_pred = estimator.run([(bound_interp, H_interp)]).result()[0].data.evs
            max_e_error = max(max_e_error, abs(e_interp_pred - e_interp_exact))

        energy_check_epochs.append(epoch + 1)
        energy_check_errors.append(max_e_error)
        lr_now = optimizer.param_groups[0]['lr']
        print(f"  Época {epoch+1:4d} | MSE={loss.item():.2e} | max ΔE={max_e_error:.2e} | lr={lr_now:.1e}")

print(f"\nEntrenamiento finalizado. MSE final: {loss_history[-1]:.2e}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

ax1.semilogy(loss_history)
ax1.set_xlabel('Época'); ax1.set_ylabel('MSE Loss')
ax1.set_title('Convergencia del MLP'); ax1.grid(True)

ax2.semilogy(energy_check_epochs, energy_check_errors, 'ro-')
ax2.axhline(y=1e-3, color='green', linestyle=':', label='Umbral 1e-3')
ax2.set_xlabel('Época'); ax2.set_ylabel('max |E_pred - E_exact|')
ax2.set_title('Validación Física (Energía)'); ax2.legend(); ax2.grid(True)

plt.tight_layout(); plt.show()

## Step 3.4: Persistencia del modelo

In [ ]:
torch.save(model.state_dict(), "mlp_hva_predictor_N6_p2.pt")
print("Modelo guardado: mlp_hva_predictor_N6_p2.pt")

---
# FASE 4: Despliegue — Warm-Start + ADAPT-VQE Restringido

## Step 4.1: Inferencia clásica ("Semilla Inteligente")

Seleccionamos $h = 1.05$, un punto no visto cerca del punto crítico $h_c = 1.0$.

In [ ]:
h_test = 1.05
H_test = build_tfim_hamiltonian(N, J, h_test)

# Ground truth por diagonalización exacta (benchmark)
evals, evecs = np.linalg.eigh(H_test.to_matrix())
e_exact_test = evals[0]
psi_exact_test = Statevector(evecs[:, 0])

# Inferencia del MLP
model.eval()
with torch.no_grad():
    h_input = torch.tensor([[h_test]], dtype=torch.float32)
    theta_pred = model(h_input).numpy().flatten()

# Evaluar energía del warm-start
bound_pred = hva_qc.assign_parameters(theta_pred)
e_pred = estimator.run([(bound_pred, H_test)]).result()[0].data.evs

print(f"h_test = {h_test}")
print(f"θ_pred = {theta_pred}")
print(f"E_exact  = {e_exact_test:.8f}")
print(f"E_pred   = {e_pred:.8f}")
print(f"ΔE       = {abs(e_pred - e_exact_test):.2e}")

## Step 4.2 + 4.3: Circuito HVA con warm-start + ADAPT-VQE restringido

Construimos un pool de operadores de Pauli (los términos no conmutantes del TFIM) y ejecutamos ADAPT-VQE con `max_iterations=2`, usando el circuito HVA con $\theta_{pred}$ como `initial_state`.

In [ ]:
from qiskit_algorithms import AdaptVQE, VQE
from qiskit_algorithms.optimizers import L_BFGS_B
from qiskit_algorithms.exceptions import AlgorithmError

# --- Pauli pool: operadores del TFIM (ZZ y X) ---
pauli_pool = [
    SparsePauliOp.from_sparse_list([("ZZ", [i, i+1], 1.0)], num_qubits=N)
    for i in range(N - 1)
] + [
    SparsePauliOp.from_sparse_list([("X", [i], 1.0)], num_qubits=N)
    for i in range(N)
]

# --- Initial state: HVA con θ_pred inyectado ---
initial_state = hva_qc.assign_parameters(theta_pred)

# --- VQE interno para ADAPT-VQE ---
vqe_solver = VQE(
    estimator=estimator,
    ansatz=QuantumCircuit(N),  # placeholder, ADAPT-VQE lo sobreescribe
    optimizer=L_BFGS_B(maxiter=100),
)

# --- ADAPT-VQE restringido ---
adapt_vqe = AdaptVQE(
    vqe_solver,
    operators=pauli_pool,
    max_iterations=2,
    gradient_threshold=1e-3,
    eigenvalue_threshold=1e-6,
    initial_state=initial_state,
)

# Si TODOS los gradientes < threshold en iteración 1, AdaptVQE lanza AlgorithmError.
# Esto es ÉXITO: el warm-start ya está tan cerca del óptimo que no necesita capas extra.
print("Ejecutando ADAPT-VQE restringido (max_iterations=2)...")
converged_at_init = False
try:
    adapt_result = adapt_vqe.compute_minimum_eigenvalue(H_test)
    print(f"\n--- Resultados ADAPT-VQE ---")
    print(f"Energía final:       {adapt_result.eigenvalue.real:.8f}")
    print(f"E_exact:             {e_exact_test:.8f}")
    print(f"ΔE:                  {abs(adapt_result.eigenvalue.real - e_exact_test):.2e}")
    print(f"Iteraciones:         {adapt_result.num_iterations}")
    print(f"Criterio de parada:  {adapt_result.termination_criterion}")
    print(f"Gradiente máx final: {abs(adapt_result.final_max_gradient):.2e}")
except AlgorithmError as e:
    if "convergence threshold" in str(e) and "first iteration" in str(e):
        converged_at_init = True
        print(f"\n✅ ADAPT-VQE: Todos los gradientes < threshold en iteración 1.")
        print(f"   El warm-start del MLP ya es (casi) óptimo — 0 capas adicionales necesarias.")
        print(f"   E_warm-start = {e_pred:.8f}")
        print(f"   E_exact      = {e_exact_test:.8f}")
        print(f"   ΔE           = {abs(e_pred - e_exact_test):.2e}")
    else:
        raise

## Step 4.4: Medición de observables locales (Caracterización de fase)

No medimos fidelidad global (prohibido en hardware). En su lugar, extraemos $\langle X_i \rangle$ y $\langle Z_i Z_{i+1} \rangle$ del estado final del ADAPT-VQE.

In [ ]:
ops_X, ops_ZZ = build_local_observables(N)

# Circuito final: si convergió en init, usar el warm-start directamente
if converged_at_init:
    final_qc = bound_pred
    e_final = e_pred
else:
    final_qc = adapt_result.optimal_circuit.assign_parameters(adapt_result.optimal_point)
    e_final = adapt_result.eigenvalue.real

sv_final = Statevector(final_qc)

# Observables del resultado final
mag_x_final = np.mean([sv_final.expectation_value(op).real for op in ops_X])
corr_zz_final = np.mean([sv_final.expectation_value(op).real for op in ops_ZZ])

# Observables exactos para h_test
mag_x_test_exact = np.mean([psi_exact_test.expectation_value(op).real for op in ops_X])
corr_zz_test_exact = np.mean([psi_exact_test.expectation_value(op).real for op in ops_ZZ])

# Observables del warm-start puro (sin ADAPT-VQE)
sv_pred = Statevector(bound_pred)
mag_x_pred = np.mean([sv_pred.expectation_value(op).real for op in ops_X])
corr_zz_pred = np.mean([sv_pred.expectation_value(op).real for op in ops_ZZ])

print(f"--- Observables locales en h={h_test} (cerca del punto crítico) ---")
print(f"{'Método':<20} {'⟨X⟩':>10} {'⟨ZZ⟩':>10}")
print(f"{'─'*42}")
print(f"{'Exact Diag':<20} {mag_x_test_exact:>10.6f} {corr_zz_test_exact:>10.6f}")
print(f"{'MLP Warm-Start':<20} {mag_x_pred:>10.6f} {corr_zz_pred:>10.6f}")
label = 'WS (grad≈0)' if converged_at_init else 'ADAPT-VQE'
print(f"{label:<20} {mag_x_final:>10.6f} {corr_zz_final:>10.6f}")
print(f"\nΔ⟨X⟩  (final vs Exact): {abs(mag_x_final - mag_x_test_exact):.2e}")
print(f"Δ⟨ZZ⟩ (final vs Exact): {abs(corr_zz_final - corr_zz_test_exact):.2e}")

## Step 4.5: Métricas de éxito y resumen

In [ ]:
# Interpretación de fase usando crossover de observables exactos de Fase 1
# En lugar de umbrales hardcoded, usamos el cruce ⟨X⟩ = ⟨ZZ⟩ del dataset exacto
crossover_idx = np.argmin(np.abs(mag_x_exact - corr_zz_exact))
h_c_finite = h_values[crossover_idx]

if h_test > h_c_finite + 0.1:
    phase = f"Paramagnética (h={h_test} > h_c≈{h_c_finite:.2f})"
elif h_test < h_c_finite - 0.1:
    phase = f"Ferromagnética (h={h_test} < h_c≈{h_c_finite:.2f})"
else:
    phase = f"Región crítica (h={h_test} ≈ h_c≈{h_c_finite:.2f})"

energy_gap = evals[1] - evals[0]
e_error = abs(e_final - e_exact_test)
e_error_pct = e_error / abs(energy_gap) * 100
n_iters = 0 if converged_at_init else adapt_result.num_iterations
criterion = 'CONVERGED (gradients ≈ 0 at init)' if converged_at_init else str(adapt_result.termination_criterion)

print("="*55)
print("       RESUMEN FINAL — PoC V2.0 Fases 3+4")
print("="*55)
print(f"  Modelo:          TFIM 1D, N={N}, J={J}, p={p_layers}")
print(f"  h_test:          {h_test} (no visto en entrenamiento)")
print(f"  h_c (finito):    {h_c_finite:.2f}")
print(f"  Fase detectada:  {phase}")
print(f"─────────────────────────────────────────────────────")
print(f"  E_exact:         {e_exact_test:.8f}")
print(f"  E_final:         {e_final:.8f}")
print(f"  ΔE:              {e_error:.2e}")
print(f"  ΔE/gap:          {e_error_pct:.4f}%")
print(f"  Gap:             {energy_gap:.6f}")
print(f"─────────────────────────────────────────────────────")
print(f"  ADAPT iteraciones:  {n_iters}")
print(f"  Criterio parada:    {criterion}")
print(f"─────────────────────────────────────────────────────")
print(f"  ⟨X⟩ error:       {abs(mag_x_final - mag_x_test_exact):.2e}")
print(f"  ⟨ZZ⟩ error:      {abs(corr_zz_final - corr_zz_test_exact):.2e}")
print("="*55)

# Validación de criterios
checks = [
    ("ΔE < 1e-3", e_error < 1e-3),
    ("ΔE/gap < 5%", e_error_pct < 5.0),
    ("ADAPT iters ≤ 2", n_iters <= 2),
    ("⟨X⟩ error < 1e-2", abs(mag_x_final - mag_x_test_exact) < 1e-2),
    ("⟨ZZ⟩ error < 1e-2", abs(corr_zz_final - corr_zz_test_exact) < 1e-2),
]
print("\nChecklist de validación:")
for name, passed in checks:
    print(f"  {'✅' if passed else '❌'} {name}")